# PYNQ-Z1 LTC2208 Dual-Channel Test
Capture both LTC2208 channels at 130 MSPS into DDR, estimate each channel frequency, and plot four cycles.

In [ ]:
from pynq import Overlay, MMIO
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

BITFILE = 'base_add.bit'
LTC2208_BASE = 0x40002000
sys.path.insert(0, str(Path('.').resolve()))
from pynqz1_diansai2_ltc2208 import LTC2208Capture, LTC2208_SAMPLE_HZ, signed_codes

overlay = Overlay(BITFILE)
dma = overlay.axi_dma_0
ltc2208 = LTC2208Capture(dma, base_addr=LTC2208_BASE)
print('Overlay:', BITFILE)
print('LTC2208:', ltc2208.status())

In [ ]:
# For the frequency/4-cycle test below, connect external signals to LTC2208 A and B.
# The final cell is an independent no-input spike diagnostic.

In [ ]:
# LTC2208 real acquisition at the full 130 MSPS.
# channel_mask: 0b01=A only, 0b10=B only, 0b11=both.
SAMPLE_COUNT = 262_144  # 2.017 ms at 130 MSPS; supports four cycles from about 2 kHz upward.
raw, adc_a, adc_b, elapsed = ltc2208.capture(
    sample_count=SAMPLE_COUNT, channel_mask=0b11, decimation=1, timeout_s=3.0
)
print('DMA elapsed = %.6f s' % elapsed)
print(ltc2208.status())

In [ ]:
# Estimate A/B dominant frequencies independently, then display four periods
# of each channel.
FREQ_MIN_HZ = 2_000       # Raise this if low-frequency drift dominates.
FREQ_MAX_HZ = 60_000_000  # LTC2208 test range, below 65 MHz Nyquist.

def estimate_tone(samples):
    signal = signed_codes(samples).astype(np.float64)
    signal -= np.mean(signal)
    n = len(signal)
    spectrum = np.abs(np.fft.rfft(signal * np.hanning(n)))
    freq_axis = np.fft.rfftfreq(n, d=1.0 / LTC2208_SAMPLE_HZ)
    valid = (freq_axis >= FREQ_MIN_HZ) & (freq_axis <= FREQ_MAX_HZ)
    if not np.any(valid):
        raise RuntimeError('No FFT bins in the requested frequency range.')
    peak_bin = np.flatnonzero(valid)[np.argmax(spectrum[valid])]
    bin_offset = 0.0
    if 0 < peak_bin < len(spectrum) - 1:
        left, center, right = spectrum[peak_bin - 1:peak_bin + 2]
        denom = left - 2.0 * center + right
        if abs(denom) > 1e-12:
            bin_offset = 0.5 * (left - right) / denom
    frequency_hz = (peak_bin + bin_offset) * LTC2208_SAMPLE_HZ / n
    return signal, frequency_hz

signals = {}
for channel_name, samples in (('A', adc_a), ('B', adc_b)):
    signal, frequency_hz = estimate_tone(samples)
    samples_per_cycle = LTC2208_SAMPLE_HZ / frequency_hz
    four_cycle_points = int(round(4.0 * samples_per_cycle))
    points = min(len(signal), max(16, four_cycle_points))
    signals[channel_name] = (signal, frequency_hz, points, four_cycle_points)
    print('LTC2208 %s: %.6f MHz (%.3f Hz), display %d samples / %.3f cycles' % (
        channel_name, frequency_hz / 1e6, frequency_hz, points, points / samples_per_cycle))
    if points < four_cycle_points:
        print('  Warning: capture is shorter than four cycles; increase SAMPLE_COUNT.')
print('FFT resolution: %.3f Hz' % (LTC2208_SAMPLE_HZ / len(adc_a)))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for axis, channel_name in zip(axes, ('A', 'B')):
    signal, frequency_hz, points, _ = signals[channel_name]
    t_us = np.arange(points) / LTC2208_SAMPLE_HZ * 1e6
    axis.plot(t_us, signal[:points], label='LTC2208 %s: %.6f MHz' % (channel_name, frequency_hz / 1e6))
    axis.set_xlabel('Time (us)')
    axis.set_ylabel('ADC signed code, DC removed')
    axis.grid(True)
    axis.legend()
plt.show()

In [ ]:
# No-input LTC2208 spike diagnostic. Disconnect both analog inputs before running.
# This cell captures again at 130 MSPS; it does not configure or test MAX5885.
SPIKE_SAMPLE_COUNT = 262_144
JUMP_LIMIT_CODES = 12_000       # Adjacent-code jump considered suspicious.
RAIL_GUARD_CODES = 1_024        # Near 0x0000 or 0xFFFF.
MAX_PRINTED_EVENTS = 12

raw_idle, idle_a, idle_b, elapsed = ltc2208.capture(
    sample_count=SPIKE_SAMPLE_COUNT, channel_mask=0b11, decimation=1, timeout_s=3.0
)
print('No-input DMA elapsed = %.6f s, samples/channel = %d (%.3f ms)' % (
    elapsed, len(idle_a), len(idle_a) / LTC2208_SAMPLE_HZ * 1e3))

def inspect_idle_channel(name, raw_codes):
    raw_codes = np.asarray(raw_codes, dtype=np.uint16)
    codes = signed_codes(raw_codes).astype(np.int32)
    jumps = np.abs(np.diff(codes))
    jump_indices = np.flatnonzero(jumps >= JUMP_LIMIT_CODES) + 1
    rail_indices = np.flatnonzero((raw_codes <= RAIL_GUARD_CODES) |
                              (raw_codes >= np.uint16(0xFFFF - RAIL_GUARD_CODES)))
    largest = np.argsort(jumps)[-min(MAX_PRINTED_EVENTS, len(jumps)):][::-1] + 1
    print('\nLTC2208 %s' % name)
    print('  raw min/max       = 0x%04X / 0x%04X' % (int(raw_codes.min()), int(raw_codes.max())))
    print('  signed min/max    = %d / %d' % (int(codes.min()), int(codes.max())))
    print('  max adjacent jump = %d codes' % int(jumps.max()))
    print('  suspicious jumps  = %d (limit %d)' % (len(jump_indices), JUMP_LIMIT_CODES))
    print('  near-rail samples = %d (guard %d codes)' % (len(rail_indices), RAIL_GUARD_CODES))
    for index in largest:
        print('    n=%6d: 0x%04X -> 0x%04X, signed %6d -> %6d, jump=%5d' % (
            index, int(raw_codes[index - 1]), int(raw_codes[index]),
            int(codes[index - 1]), int(codes[index]), int(jumps[index - 1])))
    return codes, jumps, jump_indices

idle_a_signed, idle_a_jumps, idle_a_events = inspect_idle_channel('A', idle_a)
idle_b_signed, idle_b_jumps, idle_b_events = inspect_idle_channel('B', idle_b)
print('\nCapture status:', ltc2208.status())

# Min/max envelope preserves isolated one-sample spikes in the whole-record view.
def envelope(signal, blocks=2048):
    usable = len(signal) - (len(signal) % blocks)
    shaped = signal[:usable].reshape(blocks, usable // blocks)
    time_us = np.arange(blocks) * (usable // blocks) / LTC2208_SAMPLE_HZ * 1e6
    return time_us, shaped.min(axis=1), shaped.max(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, constrained_layout=True)
for axis, name, signal, events in ((axes[0], 'A', idle_a_signed, idle_a_events),
                                    (axes[1], 'B', idle_b_signed, idle_b_events)):
    t_us, lo, hi = envelope(signal)
    axis.fill_between(t_us, lo, hi, alpha=0.65, label='min/max envelope')
    if len(events):
        axis.scatter(events / LTC2208_SAMPLE_HZ * 1e6, signal[events], c='crimson', s=14, label='suspicious jump')
    axis.set_ylabel('signed code')
    axis.set_title('LTC2208 %s, analog input disconnected' % name)
    axis.grid(True)
    axis.legend(loc='upper right')
axes[-1].set_xlabel('Time (us)')
plt.show()

# Show the first suspicious event from each channel at full sample resolution.
for name, signal, events in (('A', idle_a_signed, idle_a_events), ('B', idle_b_signed, idle_b_events)):
    if not len(events):
        print('LTC2208 %s: no jump above threshold; no local spike window to display.' % name)
        continue
    center = int(events[0])
    start, stop = max(0, center - 80), min(len(signal), center + 81)
    t_ns = (np.arange(start, stop) - center) / LTC2208_SAMPLE_HZ * 1e9
    plt.figure(figsize=(11, 3))
    plt.plot(t_ns, signal[start:stop], marker='.', markersize=3)
    plt.axvline(0, color='crimson', linestyle='--', label='jump sample')
    plt.title('LTC2208 %s local window around first suspicious jump' % name)
    plt.xlabel('Time relative to jump (ns)')
    plt.ylabel('signed code')
    plt.grid(True)
    plt.legend()
    plt.show()